<a href="https://colab.research.google.com/github/PedroCorreia3880/PedroCorreia3880.github.io/blob/main/IAC_PL02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IAC · PL02 — Partir um detector de propósito

**Inteligência Artificial em Cibersegurança** · CTeSP Cibersegurança · ISLA Gaia
Aula 2 · 2 de outubro de 2026 · Mário Amorim

---

Na PL01 olhaste para os dados. Hoje constróis um detector e passas o resto da ficha
a tentar enganar-te a ti próprio com ele, de propósito, para saberes reconhecer o
truque quando ele aparecer sem querer.

**Antes de começar:** Ficheiro → Guardar uma cópia no Drive.

## Parte 0 · Os dados

In [1]:
import urllib.request, zipfile, io, re
import pandas as pd, numpy as np

URL = 'https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
with urllib.request.urlopen(URL) as r:
    z = zipfile.ZipFile(io.BytesIO(r.read()))
bruto = z.read('SMSSpamCollection').decode('utf-8')

linhas = [l.split('\t', 1) for l in bruto.strip().split('\n')]
d = pd.DataFrame(linhas, columns=['classe', 'texto'])
d['y'] = (d.classe == 'spam').astype(int)
print(len(d), 'mensagens ·', d.y.sum(), 'spam ·', round(d.y.mean()*100, 1), '% spam')
d.head(3)

5574 mensagens · 747 spam · 13.4 % spam


,classe,texto,y
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1


## Parte A · O detector de referência (15 min)

Antes de treinar seja o que for, é preciso saber **qual é o mínimo a bater**.

### A1 · O detector que não faz nada

Escreve a previsão do detector que responde sempre «legítimo» e mede a exactidão.

In [ ]:
from sklearn.metrics import accuracy_score

previsao_burra = np.zeros(len(d), dtype=int)   # 0 = legítimo para toda a gente
print('Exactidão do detector que não faz nada:', round(accuracy_score(d.y, previsao_burra), 4))

> **Regista:** exactidão do detector que não faz nada = ______ **RESPOSTA 86,6%**
>
> Este é o número que qualquer modelo tem de bater para ter alguma utilidade.

### A2 · Características feitas à mão

Nada de vocabulários automáticos por agora. Seis perguntas que um humano faria a uma
mensagem, cada uma com resposta numérica. É isto que o modelo vai ver.

In [ ]:
GATILHOS = r'free|win|prize|urgent|claim|cash|award|txt'

F = pd.DataFrame({
    'comprimento': d.texto.str.len(),
    'digitos':     d.texto.str.count(r'\d'),
    'maiusculas':  d.texto.apply(lambda t: sum(c.isupper() for c in t) / max(len(t), 1)),
    'exclamacoes': d.texto.str.count('!'),
    'tem_url':     d.texto.str.contains(r'http|www\.|\.com', case=False).astype(int),
    'gatilho':     d.texto.str.contains(GATILHOS, case=False).astype(int),
})
F.head()

> **Pensa antes de continuar:** a coluna `gatilho` usa palavras que tu descobriste na PL01
> olhando para **todos** os dados, incluindo os que vais usar para testar. Isso é uma forma
> suave do problema da Parte B. Guarda a observação, voltamos a ela.

### A3 · Treinar e medir a sério

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

Xtr, Xte, ytr, yte = train_test_split(F, d.y, test_size=0.3, random_state=42, stratify=d.y)

modelo = LogisticRegression(max_iter=3000).fit(Xtr, ytr)
prob = modelo.predict_proba(Xte)[:, 1]
prev = (prob >= 0.5).astype(int)

vn, fp, fn, vp = confusion_matrix(yte, prev).ravel()
print(f'Verdadeiros negativos (era normal, disse normal): {vn}')
print(f'Falsos positivos      (era normal, disse spam)  : {fp}')
print(f'Falsos negativos      (era spam,   disse normal): {fn}')
print(f'Verdadeiros positivos (era spam,   disse spam)  : {vp}')
print()
print(classification_report(yte, prev, target_names=['legítimo', 'spam'], digits=3))

> **Regista:** exactidão = ______ · precisão = ______ · revocação = ______
>
> Compara com o detector que não faz nada. Quantos pontos ganhaste?
>
> E repara numa coisa: **seis características pensadas por uma pessoa**. Guarda este número,
> na aula 5 vamos compará-lo com modelos que usam o vocabulário todo.

## Parte B · Três maneiras de se enganar a si próprio (25 min)

Cada uma destas três coisas faz o número subir sem que o detector melhore.
Todas aparecem em trabalhos entregues, e nenhuma é feita de má-fé.

### B1 · Medir nos dados de treino

Treina uma árvore de decisão sem limite de profundidade e compara o que ela acerta
**nos dados com que treinou** com o que acerta em dados novos.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

arvore = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)
print('Exactidão no TREINO:', round(accuracy_score(ytr, arvore.predict(Xtr)), 4))
print('Exactidão no TESTE :', round(accuracy_score(yte, arvore.predict(Xte)), 4))
print('Diferença          :', round(accuracy_score(ytr, arvore.predict(Xtr))
                                    - accuracy_score(yte, arvore.predict(Xte)), 4))

> **Regista a diferença:** ______ pontos.
>
> A árvore decorou os exemplos. O número do treino não é um resultado, é uma medida
> de memória. **Nunca se reporta.**

### B2 · Exemplos repetidos dos dois lados

Este conjunto de dados tem mensagens exactamente iguais repetidas. Se uma cópia cai no
treino e outra no teste, o modelo está a ser avaliado com perguntas que já viu.

In [ ]:
print('Mensagens com texto repetido:', d.texto.duplicated().sum())

nos_dois = set(d.texto[Xtr.index]) & set(d.texto[Xte.index])
print('Mensagens que aparecem no treino E no teste:', len(nos_dois))

In [ ]:
# o mesmo trabalho, depois de remover as repetições
ds = d.drop_duplicates(subset=['texto']).reset_index(drop=True)
Fs = F.loc[d.drop_duplicates(subset=['texto']).index].reset_index(drop=True)

Xtr2, Xte2, ytr2, yte2 = train_test_split(Fs, ds.y, test_size=0.3, random_state=42, stratify=ds.y)
m2 = LogisticRegression(max_iter=3000).fit(Xtr2, ytr2)

print('Com repetições  (n =', len(d), '): ', round(accuracy_score(yte, prev), 4))
print('Sem repetições  (n =', len(ds), '): ', round(accuracy_score(yte2, m2.predict(Xte2)), 4))

> **Regista as duas exactidões.** Qual delas dirias a um cliente?

### B3 · Uma característica que já contém a resposta

Aqui a fuga é construída de propósito, para veres o mecanismo.

Imagina que a equipa de segurança marcou as mensagens à mão e, no processo, registou
numa coluna quais é que tinham sido **bloqueadas**. Essa coluna foi preenchida *depois*
de se saber a resposta. Se ela ficar no conjunto de dados, o modelo não tem de aprender nada.

In [ ]:
rng = np.random.RandomState(0)

# a coluna "foi bloqueada pelo analista": quase sempre igual à etiqueta, com 3% de engano
bloqueada = d.y.values.copy()
troca = rng.rand(len(d)) < 0.03
bloqueada[troca] = 1 - bloqueada[troca]

Ffuga = F.copy()
Ffuga['bloqueada_pelo_analista'] = bloqueada

Xtr3, Xte3, ytr3, yte3 = train_test_split(Ffuga, d.y, test_size=0.3, random_state=42, stratify=d.y)
m3 = LogisticRegression(max_iter=3000).fit(Xtr3, ytr3)
print('Exactidão com a coluna envenenada:', round(accuracy_score(yte3, m3.predict(Xte3)), 4))

pesos = pd.Series(m3.coef_[0], index=Ffuga.columns).sort_values(key=abs, ascending=False)
print()
print('Peso de cada característica no modelo:')
print(pesos.round(3).to_string())

> **Repara no peso da coluna envenenada** comparado com o das outras.
>
> Esta é a assinatura da fuga de informação: **uma característica com um peso muito acima
> de todas as outras e um resultado bom demais.** Sempre que virem isto, a primeira hipótese
> não é «o modelo é bom», é «esta coluna sabe a resposta».

## Parte C · O mesmo detector, num sítio a sério (15 min)

O conjunto de dados tem 13,4% de spam. Uma caixa de correio profissional com filtro à
entrada tem à volta de 1%. O detector não muda. As métricas mudam.

In [ ]:
LIMIAR = 0.15
prev_c = (prob >= LIMIAR).astype(int)
vn, fp, fn, vp = confusion_matrix(yte, prev_c).ravel()

revocacao = vp / (vp + fn)
tfp       = fp / (fp + vn)          # taxa de falsos positivos
precisao  = vp / (vp + fp)

print(f'No conjunto de teste (13,4% spam):')
print(f'  revocação = {revocacao:.3f} · taxa de falsos positivos = {tfp:.4f} · precisão = {precisao:.3f}')

In [ ]:
def precisao_a(prevalencia, revocacao, tfp, total=10_000):
    positivos = total * prevalencia
    negativos = total - positivos
    vp = revocacao * positivos
    fp = tfp * negativos
    return vp / (vp + fp)

for pv in (0.134, 0.05, 0.01, 0.001):
    print(f'  prevalência {pv:6.1%}  ->  precisão {precisao_a(pv, revocacao, tfp):6.1%}')

> **Responde no relatório:**
> 1. Porque é que a **revocação** não aparece nesta tabela?
> 2. A 1% de prevalência, de cada 10 mensagens que o detector marca como spam, quantas são legítimas?
> 3. O que é que isso significa para alguém que confia no filtro e não revê a pasta de spam?

## Parte D · Escolher o limiar a partir do custo (20 min)

Até agora o limiar foi 0,5 porque é o que vem por omissão. Agora vais escolhê-lo.

A ideia: atribuir um custo a cada tipo de erro e procurar o limiar que minimiza o custo total.
O custo não sai dos dados. Sai do problema.

In [ ]:
def custo_total(limiar, custo_fp, custo_fn):
    vn, fp, fn, vp = confusion_matrix(yte, (prob >= limiar).astype(int)).ravel()
    return custo_fp * fp + custo_fn * fn, fp, fn

limiares = np.arange(0.02, 0.99, 0.01)

CENARIOS = {
    'Caixa de correio pessoal':      (10, 1),   # chatice perder uma mensagem, spam é só incómodo
    'Clínica (marcações por email)': (50, 1),   # perder uma marcação é grave
    'Equipa de segurança (SOC)':     (1, 20),   # alarme falso custa pouco, ataque que passa custa muito
}

for nome, (cfp, cfn) in CENARIOS.items():
    custos = [custo_total(l, cfp, cfn)[0] for l in limiares]
    i = int(np.argmin(custos))
    c50, fp50, fn50 = custo_total(0.50, cfp, cfn)
    c, fp, fn = custo_total(limiares[i], cfp, cfn)
    print(f'{nome:32s} limiar {limiares[i]:.2f} · custo {c:5d} (FP={fp:3d}, FN={fn:3d})'
          f'  ·  a 0,50 o custo seria {c50}')

> **Entrega, por escrito:**
>
> 1. Escolhe **um** dos três cenários.
> 2. Diz que limiar escolheste e o custo a que isso corresponde.
> 3. Justifica em três ou quatro linhas **porque é que esse cenário tem esses custos**, sem usar
>    a palavra «exactidão».
> 4. Diz o que mudaria na tua escolha se o detector fosse usado sem qualquer revisão humana.

## Parte E · Opcional, para quem acabar cedo

Melhora o detector **sem mudar de algoritmo**: acrescenta características novas à tabela `F`
e vê se o custo do teu cenário desce.

Sugestões: número de palavras, comprimento da palavra mais longa, presença de um número de
telefone, proporção de caracteres que não são letras, número de palavras totalmente em maiúsculas.

Regra: podes olhar para os dados de **treino** à vontade. Não podes olhar para o teste.

---

### O que entregar no Moodle, até quinta-feira à noite

1. Este caderno, com as células corridas e os valores registados nos sítios pedidos.
2. Meia página com as respostas da Parte C e a justificação da Parte D.

**Leva para a aula 3** o limiar que escolheste e a razão. Vamos comparar escolhas.